<a href="https://colab.research.google.com/github/anandita-3217/EAI-Assignments/blob/main/EAI-Assignments%20/Search/SE26MAID034_PathFinding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Using Indian Metro cities as nodes calculating the implementing BFS, DFS, UCS, IDDFS

In [2]:
from collections import deque
from heapq import heappush, heappop
from itertools import count


In [7]:
Cities_Distances = [
    [0, 1419, 2121, 1499, 2107, 1453],      # Delhi
    [1419, 0, 984, 711, 1290, 1939],        # Mumbai
    [2121, 984, 0, 575, 346, 1875],         # Bengaluru
    [1499, 711, 575, 0, 633, 1492],         # Hyderabad
    [2107, 1290, 346, 633, 0, 1684],        # Chennai
    [1453, 1939, 1875, 1492, 1684, 0]       # Kolkata
]
City_Names = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai", "Kolkata"]

In [8]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def __repr__(self):
        return f"<Node {self.state}>"

In [9]:
class RouteProblem:
    def __init__(self, matrix, city_names, initial, goal):
        self.matrix = matrix
        self.city_names = city_names
        self.index_of = {name: i for i, name in enumerate(city_names)}
        self.initial = initial
        self.goal = goal

    def is_goal(self, state):
        return state == self.goal

    def actions(self, state):
        i = self.index_of[state]
        return [
            self.city_names[j]
            for j, cost in enumerate(self.matrix[i])
            if cost not in (0, None) and j != i
        ]

    def result(self, state, action):
        return action

    def action_cost(self, state, action, state2):
        i, j = self.index_of[state], self.index_of[action]
        return self.matrix[i][j]

In [5]:
def child_node(problem, parent, action):
    state = problem.result(parent.state, action)
    cost = parent.path_cost + problem.action_cost(parent.state, action, state)
    return Node(state, parent, action, cost)

In [ ]:
def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        yield child_node(problem, node, action)

In [ ]:
def solution(node):
    """Sequence of actions from initial state to this node."""
    actions = []
    while node.parent is not None:
        actions.append(node.action)
        node = node.parent
    return list(reversed(actions))

In [ ]:
def is_cycle(node, k=float('inf')):
    ancestor = node.parent
    while k > 0 and ancestor is not None:
        if ancestor.state == node.state:
            return True
        ancestor = ancestor.parent
        k -= 1
    return False

In [10]:
FAILURE = None
CUTOFF = Node("cutoff")
problem = RouteProblem(Cities_Distances, City_Names, initial="Delhi", goal="Chennai")
reached = {problem.initial: Node(problem.initial)}

In [ ]:
def best_first_search(problem, f):
    node = Node(problem.initial)
    tie = count()
    frontier = [(f(node), next(tie), node)]
    reached = {problem.initial: node}
    while frontier:
        _, _, node = heappop(frontier)
        if problem.is_goal(node.state):
            return node
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                heappush(frontier, (f(child), next(tie), child))
    return FAILURE

In [11]:
def general_search(problem, queuing_fn):
    start = Node(problem.initial)
    frontier = queuing_fn.init(start)
    reached = {problem.initial}
    while not queuing_fn.empty(frontier):
        node = queuing_fn.pop(frontier)
        if problem.is_goal(node.state):
            return node
        for child in expand(problem, node):
            if child.state not in reached:
                reached.add(child.state)
                queuing_fn.push(frontier, child)
    return None


# --- Queuing strategies ---

class FIFOQueuing:          # gives BFS
    def init(self, node): return deque([node])
    def empty(self, f): return len(f) == 0
    def pop(self, f): return f.popleft()
    def push(self, f, node): f.append(node)

class LIFOQueuing:           # gives DFS
    def init(self, node): return [node]
    def empty(self, f): return len(f) == 0
    def pop(self, f): return f.pop()
    def push(self, f, node): f.append(node)

class PriorityQueuing:       # gives UCS
    def __init__(self, f=lambda n: n.path_cost):
        self.f, self.tie = f, count()
    def init(self, node): return [(self.f(node), next(self.tie), node)]
    def empty(self, f): return len(f) == 0
    def pop(self, f): return heappop(f)[2]
    def push(self, f, node): heappush(f, (self.f(node), next(self.tie), node))

In [12]:
def depth_limited_search(problem, limit=float('inf')):
    frontier = [Node(problem.initial)]
    result = FAILURE
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        if node.depth > limit:
            result = CUTOFF
        elif not is_cycle(node):
            frontier.extend(expand(problem, node))
    return result


def iterative_deepening_search(problem):
    depth = 0
    while True:
        result = depth_limited_search(problem, depth)
        if result is not CUTOFF:
            return result
        depth += 1

In [ ]:
def bfs():
  return

In [ ]:
def dfs():
  return

In [ ]:
def ucs():
  return

In [ ]:
def iddfs():
  return